In [41]:
# verify imported data matches numbers
import pandas as pd

df = pd.read_pickle("../data/flagged.pkl") # load flagged data into a table

print(df.shape)
print(df.columns.tolist())

has_issue = df["issue_count"] >= 1
print("Numebr of rows without issues:", (len(df)-has_issue.sum()))
print("Percentage of rows with issue(s):", ((has_issue.sum()/len(df))* 100).round(2))

(1067371, 17)
['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', 'is_duplicate', 'is_cancellation', 'missing_customer', 'missing_description', 'bad_quantity', 'bad_price', 'non_product_code', 'issue_count', 'line_total']
Numebr of rows without issues: 776594
Percentage of rows with issue(s): 27.24


In [42]:
# Order	Rule	        Destination	     Reason
# 1	Duplicate	        Reject	        A copy of a row already counted
# 2	Non-product code    Reject	        Fees and adjustments, not merchandise, even when cancelled
# 3	Cancellation	    Cancellations	Real product returns, tracked separately
# 4	Bad quantity	    Reject	        Stock write-offs, not sales
# 5	Bad price	        Reject	        No money changed hands
# —	Everything else	    Sales	        Includes missing customer and missing description

# turn these rules into column "outcome"

import numpy as np # used to checl conditions

conditions = [
    df["is_duplicate"],
    df["non_product_code"],
    df["is_cancellation"],
    df["bad_quantity"],
    df["bad_price"],
]
outcomes = [
    "rejected: duplicate",
    "rejected: non product code",
    "cancellation",
    "rejected: stock adjustment",
    "rejected: bad price",
]

df["outcome"] = np.select(conditions, outcomes, default="sales")
df["outcome"].value_counts() # shows how many rows got each outcome

# print(df["outcome"].value_counts().sum() == len(df)) # check if total rows of outcome equald total rows

outcome
sales                         1003355
rejected: duplicate             34335
cancellation                    17915
rejected: non product code       5804
rejected: stock adjustment       3390
rejected: bad price              2572
Name: count, dtype: int64

In [43]:
# spot check sales
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "sales", cols].sample(5, random_state=1) # shows 5 random rows with sales label


,invoice,stock_code,description,quantity,price,customer_id
1064337,581420,20719,WOODLAND CHARLOTTE BAG,10,0.85,16358.0
996052,576617,22076,6 RIBBONS EMPIRE,2,1.25,NaN
837826,564352,22629,SPACEBOY LUNCH BOX,12,1.95,14262.0
752450,556813,22063,CERAMIC BOWL WITH STRAWBERRY DESIGN,1,5.79,NaN
358698,524163,84836,ZINC METAL HEART DECORATION,3,1.25,17524.0


In [44]:
# spot check cancellations
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "cancellation", cols].sample(5, random_state=1) # shows 5 random rows with cancel label

,invoice,stock_code,description,quantity,price,customer_id
446624,C531871,21452,TOADSTOOL MONEY BOX,-1,2.95,14105.0
288510,C517465,22061,LARGE CAKE STAND HANGING STRAWBERY,-1,9.95,17865.0
279124,C516565,22269,EGG CUP NATURAL CHICKEN,-8,1.25,12523.0
236966,C512326,85123A,WHITE HANGING HEART T-LIGHT HOLDER,-4,2.55,16001.0
730157,C554693,23203,JUMBO BAG DOILEY PATTERNS,-10,2.08,14292.0


In [45]:
# spot check rejections for bad quantity
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "rejected: stock adjustment", cols].sample(5, random_state=1) # shows 5 random rows with rej label

,invoice,stock_code,description,quantity,price,customer_id
523460,538061,21801,NaN,-38,0.0,NaN
815151,562279,21391,NaN,-26,0.0,NaN
221114,510907,90092,NaN,-123,0.0,NaN
80319,496734,82096,NaN,-1,0.0,NaN
493958,535635,22962,NaN,-100,0.0,NaN


In [46]:
# spot check rejections for bad price
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "rejected: bad price", cols].sample(5, random_state=1) # shows 5 random rows with rej label

,invoice,stock_code,description,quantity,price,customer_id
291659,517814,20724,RED SPOTTY CHARLOTTE BAG,2,0.0,NaN
80316,496733,82011E,NaN,1,0.0,NaN
718660,553521,22665,RECIPE BOX BLUE SKETCHBOOK DESIGN,2,0.0,NaN
459201,532832,22950,NaN,16,0.0,NaN
99523,498750,90002A,NaN,7,0.0,NaN


In [47]:
# create tables based on outcomes

sales = df[df["outcome"] == "sales"].copy() # keeps only rows with that label
cancelled = df[df["outcome"] == "cancellation"].copy()
rejected = df[df["outcome"].str.startswith("rejected")].copy()

print("Total sales: ", len(sales))
print("Total cancellations: ", len(cancelled))
print("Total rejected: ", len(rejected))

print("Share of sales: ", round(len(sales)/len(df)*100, 2))
print("Share of cancelled rows: ", round(len(cancelled)/len(df)*100, 2))
print("Share of rejected rows: ", round(len(rejected)/len(df)*100, 2))


Total sales:  1003355
Total cancellations:  17915
Total rejected:  46101
Share of sales:  94.0
Share of cancelled rows:  1.68
Share of rejected rows:  4.32


In [48]:
# Tidying sales table: keep necessary columns + fix types

keep = ["invoice", "stock_code", "description", "quantity", "invoice_date", "price", "customer_id", "country", "line_total"]

sales = sales[keep]
sales["customer_id"] = sales["customer_id"].astype("Int64")
sales["invoice"] = sales["invoice"].astype(str)

sales.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [49]:
# Tidying cancelled table
keep = ["invoice", "stock_code", "description", "quantity", "invoice_date", "price", "customer_id", "country", "line_total"]

cancelled = cancelled[keep]
cancelled["customer_id"] = cancelled["customer_id"].astype("Int64")
cancelled["invoice"] = cancelled["invoice"].astype(str)

cancelled.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4


In [50]:
# Tidying rejected table
keep = ["invoice", "stock_code", "description", "quantity", "invoice_date", "price", "customer_id", "country", "line_total", "outcome"]

rejected = rejected[keep]
rejected["customer_id"] = rejected["customer_id"].astype("Int64")
rejected["invoice"] = rejected["invoice"].astype(str)

rejected.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,outcome
89,489439,POST,POSTAGE,3,2009-12-01 09:28:00,18.0,12682,France,54.0,rejected: non product code
126,489444,POST,POSTAGE,1,2009-12-01 09:55:00,141.0,12636,USA,141.0,rejected: non product code
173,489447,POST,POSTAGE,1,2009-12-01 10:10:00,130.0,12362,Belgium,130.0,rejected: non product code
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,-0.0,rejected: stock adjustment
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,-0.0,rejected: stock adjustment


In [52]:
# Save table and cleaning summary

# save three tables as CSV so SQL and BI can read 
sales.to_csv("../data/sales.csv", index=False)
cancelled.to_csv("../data/cancelled.csv", index=False)
rejected.to_csv("../data/rejected.csv", index=False)

# cleaning summary to reports folder
summary = df["outcome"].value_counts().to_frame() # turns counts into a table with a column named count
summary["pct_of_total"] = (summary["count"]/len(df) * 100).round(2) # adds column showing pct of each type
summary.to_csv("../reports/cleaning_summary.csv", index_label="outcome")
summary

# ensure accurate data before being loaded into postgresql
check = pd.read_csv("../data/sales.csv")
print("Saved file complete:", len(check) == len(sales))

Saved file complete: True
